## 任务部分 (b)

### CVRP 运筹优化模型构建

本模型通过引入 **流量变量 (Flow variables)** 来有效处理子回路消除约束（Subtour Elimination Constraints），并实时跟踪车辆的剩余载货量。

---

#### 1. 集合与索引
* $V = \{0, 1, \dots, 25\}$：所有节点的集合（其中 $0$ 为配送中心，其余为零售点）。
* $V_C = \{1, \dots, 25\}$：零售点（客户）集合。
* $K = \{1, 2\}$：车辆类型集合。
* $A = \{(i,j) \in V \times V \mid i \neq j\}$：可行弧（路径）集合。

#### 2. 参数定义
* $c^k$：车辆类型 $k$ 的单位距离运输成本 [元/km]。
* $s_{ij}$：节点 $i$ 与 $j$ 之间的欧几里得距离 [km]。
* $d_i$：零售点 $i$ 的货物需求量。
* $q^k$：车辆类型 $k$ 的最大载重容量。

#### 3. 决策变量
* $x_{ij}^k \in \{0, 1\}$：**二进制变量**。如果车辆 $k$ 经过弧 $(i, j)$ 则取值为 1，否则为 0。
* $f_{ij}^k \ge 0$：**连续变量**。表示车辆 $k$ 在弧 $(i, j)$ 上承载的货物流量。

#### 4. 目标函数
**最小化总运输成本：**
$$
\min \sum_{k \in K} \sum_{(i,j) \in A} c^k \cdot s_{ij} \cdot x_{ij}^k
$$

#### 5. 约束条件

**A. 车辆启动约束**
确保每种类型的车辆至少有一辆从配送中心出发：
$$
\sum_{j \in V_C} x_{0j}^k \ge 1 \quad \forall k \in K
$$

**B. 客户访问唯一性**
每个零售点必须且仅能由一辆车访问一次：
$$
\sum_{k \in K} \sum_{i \in V, i \neq j} x_{ij}^k = 1 \quad \forall j \in V_C
$$

**C. 车辆流平衡（路径连续性）**
进入某节点的车辆必须从该节点离开：
$$
\sum_{i \in V, i \neq j} x_{ij}^k = \sum_{l \in V, l \neq j} x_{jl}^k \quad \forall j \in V, \forall k \in K
$$

**D. 货物流平衡（需求覆盖）**
节点的流入量与流出量之差必须等于该点的需求量：
$$
\sum_{i \in V, i \neq j} f_{ij}^k - \sum_{l \in V, l \neq j} f_{jl}^k = d_j \cdot \sum_{i \in V, i \neq j} x_{ij}^k \quad \forall j \in V_C, \forall k \in K
$$

**E. 容量与耦合约束**
确保弧段流量不超过车型载重上限，且仅在车辆通过该弧段时允许产生流量：
$$
0 \le f_{ij}^k \le q^k \cdot x_{ij}^k \quad \forall (i,j) \in A, \forall k \in K
$$